# VieNeu-TTS — thử đọc tiếng Việt
Chọn **Runtime → Change runtime type → T4 GPU**, rồi **Run all**.
Sửa `TEXTS` ở cell 2 để thử câu khác. Audio WAV được lưu tại `/content/vieneu_test` và nghe ngay bên dưới.

Dùng một phiên Colab riêng cho notebook này. Đây là bản test vài câu; phần đọc đường dẫn dataset sẽ thêm sau.
[Nguồn VieNeu-TTS](https://github.com/pnnbao97/VieNeu-TTS)

In [ ]:
# 1. Cài thư viện vào Python đang chạy notebook.
%pip install -q "vieneu==3.8.3" "transformers==4.57.6" safetensors

In [ ]:
# 2. Nhập câu muốn đọc.
TEXTS = [
    "Xin chào, đây là câu thử giọng nói tiếng Việt.",
    "Nhắn cho mẹ là tối nay con về muộn mười phút.",
]
VOICE = None  # None: tự chọn giọng có sẵn; hoặc điền tên giọng in ở cell 3.

In [ ]:
# 3. Tải model lên GPU. Lần đầu cần chờ tải weights.
import torch
from vieneu import Vieneu

assert torch.cuda.is_available(), "Hãy chọn T4 GPU trong Runtime → Change runtime type."
tts = Vieneu(mode="v3turbo", backend="pytorch", device="cuda", dtype="float16")
voices = tts.list_preset_voices()
voice = VOICE or voices[0][1]
print("GPU:", torch.cuda.get_device_name(0))
print("Giọng có sẵn:", [name for label, name in voices])
print("Giọng đang dùng:", voice)

In [ ]:
# 4. Tạo WAV và nghe thử. Sửa TEXTS rồi chạy lại cell này để thử tiếp.
from pathlib import Path
from IPython.display import Audio, display

output_dir = Path("/content/vieneu_test")
output_dir.mkdir(parents=True, exist_ok=True)
for i, text in enumerate(TEXTS, 1):
    with torch.inference_mode():
        audio = tts.infer(text, voice=voice)
    path = output_dir / f"test_{i:02d}.wav"
    tts.save(audio, str(path))
    print(text, "→", path)
    display(Audio(filename=str(path)))